# Backtrader + Subgrounds (Uniswap v3)
本笔记本使用 Subgrounds 抓取 Uniswap v3 swaps，聚合成 1min/5min K 线，并用 Backtrader 进行离线回测（含 Gas / 滑点 / IL 影响）。

In [ ]:
from pathlib import Path
import os
import sys

root_dir = Path("..")
sys.path.append(str(root_dir / "src" / "backtests"))

from subgrounds_klines import SwapQueryConfig, fetch_swaps_subgrounds, build_klines, make_sample_swaps
from backtrader_uniswap import CostConfig, run_backtest

In [ ]:
SUBGRAPH_ENDPOINT = os.getenv("UNISWAP_V3_ARBITRUM_ENDPOINT", "")
POOL_ADDRESS = os.getenv("UNISWAP_V3_ARBITRUM_ETH_USDC_POOL_ADDRESS", "")
GRAPH_API_KEY = os.getenv("GRAPH_API_KEY", "")

use_sample = not (SUBGRAPH_ENDPOINT and POOL_ADDRESS)

if use_sample:
    swaps_df = make_sample_swaps()
else:
    if GRAPH_API_KEY:
        headers = {"Authorization": f"Bearer {GRAPH_API_KEY}"}
    else:
        headers = None
    cfg = SwapQueryConfig(endpoint=SUBGRAPH_ENDPOINT, pool_address=POOL_ADDRESS, headers=headers)
    swaps_df = fetch_swaps_subgrounds(cfg)

swaps_df.head()

In [ ]:
bars_1m = build_klines(swaps_df, "1min")
bars_5m = build_klines(swaps_df, "5min")

bars_1m.head(), bars_5m.head()

In [ ]:
cost_cfg = CostConfig(fee_rate=0.0005, gas_usd=1.5, slippage_bps=3.0)

if not bars_1m.empty:
    cerebro_1m = run_backtest(bars_1m, cash=10000.0, cost_cfg=cost_cfg)
    print("Final equity (1m):", round(cerebro_1m.broker.getvalue(), 2))

if not bars_5m.empty:
    cerebro_5m = run_backtest(bars_5m, cash=10000.0, cost_cfg=cost_cfg)
    print("Final equity (5m):", round(cerebro_5m.broker.getvalue(), 2))